# eg7 (v2) — SymPy → NumPy → (A) Matplotlib / (B) GeoGebra `LineGraph` through the `eval` verb

v1 (`eg7_plotting.ipynb`) sent the two sample lists with three `await ggb.command(...)` round trips and captured a PNG with `ggb.function('getPNGBase64', …)`.
v2: the three commands travel in one `Eval` (one RPC), the result is read back as data (`xml_out` → `ConstructionIO.from_xml` / `element_irs`), and there is no `function(...)` escape hatch — `getPNGBase64` is outside the closed subset of 8 host verbs (C1), so the PNG capture is recorded as an allow-list question, not called.

In [ ]:
import sys, time; sys.path.insert(0, '/Users/manabu/work/ggblab-replay')
import sympy as sp, numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from ggblab_extra import ConstructionIO, element_irs
import ggblab.host.html_host as H; H.DEPLOY = 'https://cdn.geogebra.org/apps/deployggb.js'
from ggblab import GeoGebra

## 1. symbolic function → numeric samples (SymPy `lambdify`, NumPy)

In [ ]:
x = sp.symbols('x')
f_expr = sp.sin(x) + 0.2 * x**2
f_num = sp.lambdify(x, f_expr, modules='numpy')
xs = np.linspace(-3, 3, 201); ys = f_num(xs)
print('f =', f_expr, '| samples', xs.size, '| f(0) =', float(f_num(0.0)), '| max', float(ys.max()))

## 2. (A) Matplotlib — static

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4)); ax.plot(xs, ys, label=str(f_expr)); ax.set_title('SymPy -> Matplotlib (static)'); ax.set_xlabel('x'); ax.set_ylabel('f(x)'); ax.legend()
fig.savefig('/tmp/eg7_static.png', dpi=72); plt.close(fig); print('static figure written')

## 3. (B) GeoGebra — two lists + `LineGraph(xList, yList)` in ONE `eval` (v1: three awaits)

In [ ]:
g = GeoGebra(appName='graphing', showAlgebraInput=True); g

In [ ]:
xs_str = ','.join(f'{float(v):.6f}' for v in xs); ys_str = ','.join(f'{float(v):.6f}' for v in ys)
t0 = time.time()
labels = g.command(f'data_x = {{{xs_str}}}', f'data_y = {{{ys_str}}}', 'graph = LineGraph(data_x, data_y)', timeout=60)
print('LABELS', labels, 'in', round(time.time() - t0, 2), 's; payload', len(xs_str) + len(ys_str), 'chars')

## 4. read it back as data (`xml_out` once → DataFrame + the XML's own IR); no `getValueString`

In [ ]:
xml = g.xml(timeout=60); df = ConstructionIO.from_xml(xml); irs = element_irs(xml)
print(df.filter(df['Name'].is_in(['data_x', 'data_y', 'graph'])).select(['Name', 'Type', 'Command']))
n_x = irs['data_x'].expression.count(',') + 1 if irs['data_x'].expression else None
print('graph:', irs['graph'].type, irs['graph'].command, '| data_x entries in the XML:', n_x, '| ROWS', df.height)

## 5. PNG capture — v1 `getPNGBase64` is not one of the 8 verbs (eval / new / delete / xml_in / xml_out / value / kind / listen)

Adding a verb is an interface change on the allow-list (C1, code-graph instrument M2) — teacher's call; nothing is called here.

In [ ]:
print('DONE')